# Hate Speech Classification with Robustness Testing
**Dataset:** HateXplain (Mathew et al., 2021)  
**Model:** TF-IDF + Logistic Regression for 3-class classification (hate / offensive / normal)  
**Focus:** Robustness to text obfuscation — leet-speak, punctuation insertion, character repetition

---

## Configuration

All parameters are set here. Edit this cell to change the experiment.

In [ ]:
# ── Configuration ────────────────────────────────────────
MAX_FEATURES = 20000              # max TF-IDF vocabulary size
NGRAM_RANGE  = (1, 2)             # unigrams + bigrams
MIN_DF       = 3                  # minimum document frequency
RANDOM_SEED  = 42

# ── Paths ─────────────────────────────────────────────────
DRIVE_DIR    = "/content/drive/MyDrive/hate_speech_project"

# ── Label mapping ─────────────────────────────────────────
LABEL_MAP    = {"hatespeech": 0, "offensive": 1, "normal": 2}
LABEL_NAMES  = {0: "hate", 1: "offensive", 2: "normal"}
COLORS       = {"hate": "#e63946", "offensive": "#f4a261", "normal": "#2a9d8f"}

## Imports & Shared Utilities

In [ ]:
import json, re, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score)

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words("english"))

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("✓ All imports done")

In [ ]:
# ── Shared: Obfuscation functions ────────────────────────
LEET_MAP = {'a':'4','e':'3','i':'1','o':'0','s':'5','t':'7','g':'9','b':'8'}

def leet_speak(text, rate=0.4):
    """Replace letters with leet equivalents (e.g. hate → h4t3)."""
    return "".join(LEET_MAP[c.lower()] if c.lower() in LEET_MAP
                   and random.random() < rate else c for c in text)

def insert_punctuation(text, rate=0.3):
    """Insert punctuation between chars of random words (e.g. hate → h.a.t.e)."""
    words = text.split()
    for i, w in enumerate(words):
        if len(w) > 3 and random.random() < rate:
            words[i] = random.choice([".","-","_","*"]).join(list(w))
    return " ".join(words)

def char_repeat(text, rate=0.3):
    """Randomly repeat a character in random words (e.g. hate → haate)."""
    words = text.split()
    for i, w in enumerate(words):
        if len(w) > 3 and random.random() < rate:
            idx = random.randint(1, len(w)-2)
            words[i] = w[:idx] + w[idx]*random.randint(2,4) + w[idx+1:]
    return " ".join(words)

def combined_obfuscation(text):
    """Apply all three obfuscation strategies."""
    return char_repeat(insert_punctuation(leet_speak(text, 0.3), 0.2), 0.2)

OBFUSCATION_FNS = {
    "leet_speak":  leet_speak,
    "punctuation": insert_punctuation,
    "char_repeat": char_repeat,
    "combined":    combined_obfuscation,
}

# Preview
random.seed(RANDOM_SEED)
s = "I hate those people they should all leave"
print(f"Original    : {s}")
print(f"Leet        : {leet_speak(s)}")
print(f"Punctuation : {insert_punctuation(s)}")
print(f"Char repeat : {char_repeat(s)}")
print(f"Combined    : {combined_obfuscation(s)}")

In [ ]:
# ── Shared: prediction & robustness ──────────────────────
def predict_with_probs(vectorizer, model, texts):
    """Predict classes and return probabilities."""
    X = vectorizer.transform(texts)
    preds = model.predict(X)
    probs = model.predict_proba(X)
    return np.array(preds), np.array(probs)


def robustness_eval(vectorizer, model, texts, labels, reference_f1=None):
    """Evaluate model on clean + all obfuscated variants."""
    random.seed(RANDOM_SEED)
    results, obf_cache = {}, {"clean": texts}
    preds_clean, _ = predict_with_probs(vectorizer, model, texts)
    results["clean"] = f1_score(labels, preds_clean, average="macro")
    ref = reference_f1 or results["clean"]
    print(f"  clean          → F1: {results['clean']:.4f}")
    for name, fn in OBFUSCATION_FNS.items():
        obf = [fn(t) for t in texts]
        obf_cache[name] = obf
        p, _ = predict_with_probs(vectorizer, model, obf)
        f1   = f1_score(labels, p, average="macro")
        results[name] = f1
        print(f"  {name:<16} → F1: {f1:.4f}  (drop: {ref-f1:+.4f})")
    return results, obf_cache, preds_clean


def plot_cm(labels, preds, title="", cmap="Blues", save_path="cm.png"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["hate","offensive","normal"],
                yticklabels=["hate","offensive","normal"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix {title}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved {save_path}")

print("✓ All shared utilities defined")

---
## Task 1 — Problem Definition & Dataset Understanding

**Task:** Classify social media posts into three categories — *hate speech*, *offensive language*, and *normal*.  
This is an NLP classification problem because it requires understanding context, implicit meaning, and social norms from raw text — not just surface-level keywords.

**Dataset:** [HateXplain](https://github.com/hate-alert/HateXplain) — ~20,000 posts from Twitter and Gab, annotated by 3 workers each. Labels are resolved via majority vote. Posts with no majority (~2–3%) are discarded.

**Known limitations:** class imbalance; annotation subjectivity; English-only; no demographic information on annotators.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/"

def download_json(filename):
    with urllib.request.urlopen(BASE_URL + filename) as r:
        return json.loads(r.read().decode())

print("Downloading dataset from GitHub...")
data       = download_json("dataset.json")
split_info = download_json("post_id_divisions.json")
print(f"✓ Total posts: {len(data)} | Splits: {list(split_info.keys())}")

# Inspect one example
sample_id = list(data.keys())[0]
print(f"\n── Sample post ({sample_id}) ──")
print(json.dumps(data[sample_id], indent=2)[:600])

In [ ]:
def majority_vote(label_list):
    count = Counter(label_list)
    top   = count.most_common(1)[0]
    return top[0] if top[1] >= 2 else None

def build_dataframe(post_ids):
    rows = []
    for pid in post_ids:
        if pid not in data: continue
        post   = data[pid]
        labels = [a["label"] for a in post["annotators"]]
        vote   = majority_vote(labels)
        if vote is None: continue
        rows.append({"post_id": pid,
                     "text":       " ".join(post["post_tokens"]),
                     "label":      LABEL_MAP[vote],
                     "label_name": LABEL_NAMES[LABEL_MAP[vote]]})
    return pd.DataFrame(rows)

train_df = build_dataframe(split_info["train"])
val_df   = build_dataframe(split_info["val"])
test_df  = build_dataframe(split_info["test"])

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
print(f"\nLabel distribution (train):\n{train_df['label_name'].value_counts()}")

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)
print("\n✓ Saved train.csv / val.csv / test.csv")

---
## Task 2 — Exploratory Data Analysis

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

# ── Class distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [("Train",train_df),("Validation",val_df),("Test",test_df)]):
    counts = df["label_name"].value_counts().reindex(["hate","offensive","normal"])
    bars   = ax.bar(counts.index, counts.values,
                    color=[COLORS[l] for l in counts.index], edgecolor="white")
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Class"); ax.set_ylabel("Count")
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                str(v), ha="center", fontsize=10)
    ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Class Distribution across Splits", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_class_distribution.png")

In [ ]:
# ── Text length analysis ──────────────────────────────────
train_df["token_count"] = train_df["text"].apply(lambda x: len(str(x).split()))
train_df["char_count"]  = train_df["text"].apply(lambda x: len(str(x)))

print("── Token count by class ──")
print(train_df.groupby("label_name")["token_count"].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color in COLORS.items():
    axes[0].hist(train_df[train_df["label_name"]==label]["token_count"],
                 bins=40, alpha=0.6, label=label, color=color, edgecolor="none")
axes[0].set_title("Token Count Distribution", fontweight="bold")
axes[0].set_xlabel("Tokens"); axes[0].set_ylabel("Frequency"); axes[0].legend()
axes[0].spines[["top","right"]].set_visible(False)

sns.boxplot(data=train_df, x="label_name", y="token_count", palette=COLORS,
            order=["hate","offensive","normal"], ax=axes[1], hue="label_name")
axes[1].set_title("Token Count Boxplot", fontweight="bold")
axes[1].spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("eda_text_length.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_text_length.png")

In [ ]:
# ── Top-20 tokens per class ───────────────────────────────
def get_top_tokens(texts, n=20):
    tokens = []
    for text in texts:
        tokens.extend([w.lower() for w in str(text).split()
                       if w.isalpha() and w.lower() not in STOPWORDS and len(w) > 2])
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, color) in zip(axes, COLORS.items()):
    top = get_top_tokens(train_df[train_df["label_name"]==label]["text"])
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor="none")
    ax.set_title(f"Top 20 Tokens — {label.capitalize()}", fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Most Frequent Tokens by Class (NLTK stopwords removed)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_top_tokens.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_top_tokens.png")

In [ ]:
# ── Word clouds ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, cmap) in zip(axes, {"hate":"Reds","offensive":"Oranges","normal":"Greens"}.items()):
    text  = " ".join(train_df[train_df["label_name"]==label]["text"].tolist())
    clean = " ".join([w for w in text.split()
                      if w.lower() not in STOPWORDS and w.isalpha() and len(w) > 2])
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap=cmap, max_words=100, collocations=False).generate(clean)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(label.capitalize(), fontsize=13, fontweight="bold")
plt.suptitle("Word Clouds by Class", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_wordclouds.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_wordclouds.png")

In [ ]:
# ── Summary statistics ────────────────────────────────────
print("══════════════════════════════════════════")
print("        EDA SUMMARY — TRAIN SPLIT")
print("══════════════════════════════════════════")
total = len(train_df)
for label in ["hate","offensive","normal"]:
    n   = (train_df["label_name"]==label).sum()
    avg = train_df[train_df["label_name"]==label]["token_count"].mean()
    print(f"  {label:<12} | n={n:>4} ({n/total*100:.1f}%) | avg tokens={avg:.1f}")
print(f"  {'TOTAL':<12} | n={total}")
print("══════════════════════════════════════════")
print(f"  Overall avg tokens : {train_df['token_count'].mean():.1f}")
print(f"  Overall avg chars  : {train_df['char_count'].mean():.1f}")
print(f"  Max token count    : {train_df['token_count'].max()}")
print(f"  Min token count    : {train_df['token_count'].min()}")

---
## Task 3 — Data Preprocessing

Preprocessing is minimal: replace URLs and @mentions with placeholders, strip `#` from hashtags. TF-IDF operates on raw token frequencies, so no subword tokenization is needed.

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "[URL]",  text)
    text = re.sub(r"@\w+",            "[USER]", text)
    text = re.sub(r"#(\w+)",           r"\1",   text)
    text = re.sub(r"\s+",              " ",      text).strip()
    return text

for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["text"].apply(clean_text)

print("── Before / After cleaning ──")
for i in range(2):
    print(f"\nOriginal : {train_df['text'].iloc[i]}")
    print(f"Cleaned  : {train_df['clean_text'].iloc[i]}")

In [ ]:
# ── Save clean CSVs ───────────────────────────────────────
cols = ["post_id","text","clean_text","label","label_name"]
train_df[cols].to_csv("train_clean.csv", index=False)
val_df[cols].to_csv("val_clean.csv",     index=False)
test_df[cols].to_csv("test_clean.csv",   index=False)
print("✓ Saved train_clean.csv / val_clean.csv / test_clean.csv")

---
## Task 3b — TF-IDF Vectorization

In [ ]:
# ── TF-IDF Vectorization ─────────────────────────────────
train_df = pd.read_csv("train_clean.csv")
val_df   = pd.read_csv("val_clean.csv")
test_df  = pd.read_csv("test_clean.csv")

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_val   = vectorizer.transform(val_df["clean_text"])
X_test  = vectorizer.transform(test_df["clean_text"])

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

print(f"✓ Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"  TF-IDF matrix : {X_train.shape}")
print(f"  Features      : unigrams + bigrams, min_df={MIN_DF}")

---
## Task 4a — Baseline Training (TF-IDF + Logistic Regression)

**Model:** Logistic Regression with TF-IDF features (unigrams + bigrams). A linear model that serves as a classical ML baseline for comparison with deep learning and transformer-based approaches.

**Training:** No iterative epochs — sklearn fits in one shot using L-BFGS solver.

In [ ]:
# ── Baseline: Logistic Regression ────────────────────────
model_4a = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    solver="lbfgs",
    multi_class="multinomial"
)
model_4a.fit(X_train, y_train)

# Validation
val_preds = model_4a.predict(X_val)
val_f1 = f1_score(y_val, val_preds, average="macro")
print(f"✓ Baseline trained | Val macro F1: {val_f1:.4f}")

In [ ]:
# ── Baseline — Test Set Results ──────────────────────────
preds_4a = model_4a.predict(X_test)

print("── Task 4a — Test Set Results ──")
print(classification_report(y_test, preds_4a,
                             target_names=["hate","offensive","normal"], digits=4))
plot_cm(y_test, preds_4a,
        title="— LR Baseline", cmap="Blues",
        save_path="cm_4a.png")

---
## Task 4b — Improved Model (Balanced Class Weights)

Same Logistic Regression model but with `class_weight='balanced'` to give higher weight to minority classes (hate, offensive) during training.

In [ ]:
# ── Improved: LR with balanced class weights ─────────────
model_4b = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    solver="lbfgs",
    multi_class="multinomial",
    class_weight="balanced"
)
model_4b.fit(X_train, y_train)

# Validation
val_preds_4b = model_4b.predict(X_val)
val_f1_4b = f1_score(y_val, val_preds_4b, average="macro")
print(f"✓ Improved trained | Val macro F1: {val_f1_4b:.4f}")

In [ ]:
# ── Improved — Test Set Results ──────────────────────────
preds_4b = model_4b.predict(X_test)

print("── Task 4b — Test Set Results ──")
print(classification_report(y_test, preds_4b,
                             target_names=["hate","offensive","normal"], digits=4))
plot_cm(y_test, preds_4b,
        title="— LR Improved", cmap="Greens",
        save_path="cm_4b.png")

---
## Task 5 — Robustness to Obfuscation

In [ ]:
# ── Baseline robustness ──────────────────────────────────
test_texts  = test_df["clean_text"].tolist()
test_labels = test_df["label"].tolist()

print("── Baseline robustness ──")
results_4a, obf_cache, preds_clean_4a = robustness_eval(
    vectorizer, model_4a, test_texts, test_labels)

In [ ]:
# ── Robustness bar chart — baseline ──────────────────────
fig, ax = plt.subplots(figsize=(8,4))
bar_colors = ["#2a9d8f"] + ["#e63946"]*4
bars = ax.bar(results_4a.keys(), results_4a.values(),
              color=bar_colors, edgecolor="white")
ax.axhline(results_4a["clean"], color="gray", linestyle="--", alpha=0.5, label="Baseline clean")
for bar, v in zip(bars, results_4a.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("Macro F1"); ax.set_ylim(0, 1.0)
ax.set_title("Robustness to Obfuscation — Baseline", fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("robustness_baseline.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ── Per-class breakdown — baseline ───────────────────────
print("── Per-class F1 breakdown — Baseline ──")
header = f"{'Condition':<18} {'hate':>8} {'offensive':>10} {'normal':>8} {'macro':>8}"
print(header); print("─"*len(header))
for name, txt_list in obf_cache.items():
    p, _ = predict_with_probs(vectorizer, model_4a, txt_list)
    f1s  = f1_score(test_labels, p, average=None)
    macro = f1_score(test_labels, p, average="macro")
    print(f"  {name:<16} {f1s[0]:8.4f} {f1s[1]:10.4f} {f1s[2]:8.4f} {macro:8.4f}")

### 5b — Improved model robustness

In [ ]:
print("── Improved robustness ──")
results_4b, _, preds_clean_4b = robustness_eval(
    vectorizer, model_4b, test_texts, test_labels,
    reference_f1=results_4a["clean"])

In [ ]:
# ── Side-by-side comparison chart ────────────────────────
keys  = list(results_4a.keys())
x     = np.arange(len(keys))
width = 0.35
fig, ax = plt.subplots(figsize=(10,5))
b1 = ax.bar(x-width/2, [results_4a[k] for k in keys], width,
            label="Baseline", color="#457b9d", alpha=0.85)
b2 = ax.bar(x+width/2, [results_4b[k] for k in keys], width,
            label="Improved", color="#2a9d8f", alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(keys)
ax.set_ylabel("Macro F1"); ax.set_ylim(0, 1.0)
ax.set_title("Baseline vs Improved — Robustness to Obfuscation", fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("robustness_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ── Δ Summary ────────────────────────────────────────────
print("── Δ Summary ──")
header = f"{'Condition':<20} {'Baseline':>10} {'Improved':>10} {'Δ':>10}"
print(header); print("─"*len(header))
for name in results_4a:
    b = results_4a[name]
    imp = results_4b[name]
    print(f"  {name:<18} {b:10.4f} {imp:10.4f} {imp-b:+10.4f}")

### 5c — Qualitative failure analysis

In [ ]:
# Find posts correct on clean but wrong after combined obfuscation
random.seed(RANDOM_SEED)
combined_texts = obf_cache["combined"]
preds_combined, _ = predict_with_probs(vectorizer, model_4a, combined_texts)

failures = []
for i, (cp, op, tl) in enumerate(zip(preds_clean_4a, preds_combined, test_labels)):
    if cp == tl and op != tl:
        failures.append({
            "idx": i,
            "text": test_texts[i][:80],
            "obfuscated": combined_texts[i][:80],
            "true": LABEL_NAMES[tl],
            "clean_pred": LABEL_NAMES[cp],
            "obf_pred": LABEL_NAMES[op]
        })

print(f"Posts flipped by combined obfuscation: {len(failures)} / {len(test_labels)} "
      f"({100*len(failures)/len(test_labels):.1f}%)\n")
for ex in random.sample(failures, min(8, len(failures))):
    print(f"  True: {ex['true']:<12} Clean→{ex['clean_pred']:<10} Obf→{ex['obf_pred']}")
    print(f"    {ex['text']}")
    print(f"    → {ex['obfuscated']}\n")

---
## Threshold Tuning — Hate Class Recall Analysis

The baseline model uses argmax (highest probability wins). By lowering the decision threshold for the `hate` class, we can increase recall at the cost of precision. This is desirable in content moderation, where missing a hateful post (false negative) is worse than flagging a normal one (false positive).

In [ ]:
# ── Threshold sweep on test set (baseline model) ─────────
_, all_probs = predict_with_probs(vectorizer, model_4a, test_texts)

thresholds = np.arange(0.15, 0.55, 0.025)
results_th = []

for th in thresholds:
    preds_th = []
    for prob in all_probs:
        if prob[0] >= th:
            preds_th.append(0)
        else:
            preds_th.append(prob.argmax())
    
    prec = precision_score(test_labels, preds_th, labels=[0], average=None)[0]
    rec  = recall_score(test_labels, preds_th, labels=[0], average=None)[0]
    f1_h = f1_score(test_labels, preds_th, labels=[0], average=None)[0]
    f1_m = f1_score(test_labels, preds_th, average="macro")
    results_th.append({"threshold": th, "hate_precision": prec, 
                        "hate_recall": rec, "hate_f1": f1_h, "macro_f1": f1_m})

df_th = pd.DataFrame(results_th)
print(df_th.to_string(index=False, float_format="%.3f"))

In [ ]:
# ── Precision-Recall tradeoff plot ────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(df_th["threshold"], df_th["hate_recall"], "o-", color="#e63946", 
         linewidth=2, label="Hate Recall")
ax1.plot(df_th["threshold"], df_th["hate_precision"], "o-", color="#457b9d", 
         linewidth=2, label="Hate Precision")
ax1.plot(df_th["threshold"], df_th["hate_f1"], "o--", color="#f4a261", 
         linewidth=2, label="Hate F1")

ax1.axvline(1/3, color="gray", linestyle=":", alpha=0.7, label="Default (argmax ≈ 0.33)")

ax1.set_xlabel("Hate Class Threshold", fontweight="bold")
ax1.set_ylabel("Score", fontweight="bold")
ax1.set_title("Threshold Tuning — Hate Class Precision vs Recall", fontweight="bold")
ax1.legend(loc="center right")
ax1.set_ylim(0.4, 1.0)
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved threshold_tuning.png")

In [ ]:
# ── Compare default vs tuned threshold ────────────────────
default_row = df_th.iloc[(df_th["threshold"] - 1/3).abs().argsort().iloc[0]]
best_recall_row = df_th[df_th["hate_recall"] >= 0.90].iloc[-1] if (df_th["hate_recall"] >= 0.90).any() else df_th.iloc[df_th["hate_recall"].argmax()]

print("── Default (argmax) ──")
print(f"   Threshold     : {default_row['threshold']:.3f}")
print(f"   Hate Precision: {default_row['hate_precision']:.3f}")
print(f"   Hate Recall   : {default_row['hate_recall']:.3f}")
print(f"   Hate F1       : {default_row['hate_f1']:.3f}")
print(f"   Macro F1      : {default_row['macro_f1']:.3f}")

print(f"\n── Tuned (recall ≥ 0.90) ──")
print(f"   Threshold     : {best_recall_row['threshold']:.3f}")
print(f"   Hate Precision: {best_recall_row['hate_precision']:.3f}")
print(f"   Hate Recall   : {best_recall_row['hate_recall']:.3f}")
print(f"   Hate F1       : {best_recall_row['hate_f1']:.3f}")
print(f"   Macro F1      : {best_recall_row['macro_f1']:.3f}")

delta_rec  = best_recall_row['hate_recall'] - default_row['hate_recall']
delta_prec = best_recall_row['hate_precision'] - default_row['hate_precision']
print(f"\n── Tradeoff ──")
print(f"   Recall gain   : +{delta_rec:.3f}")
print(f"   Precision cost: {delta_prec:.3f}")

---
## Save to Google Drive

In [ ]:
import joblib
from google.colab import drive
drive.mount("/content/drive")
import os

os.makedirs(DRIVE_DIR, exist_ok=True)

joblib.dump(model_4a, os.path.join(DRIVE_DIR, "lr_baseline.pkl"))
joblib.dump(model_4b, os.path.join(DRIVE_DIR, "lr_improved.pkl"))
joblib.dump(vectorizer, os.path.join(DRIVE_DIR, "tfidf_vectorizer.pkl"))
print("✓ Models and vectorizer saved to Drive")